> **Superseded exploratory notebook — not used for any result in the manuscript.**
>
> This is the earlier protocol mentioned in Section 3.5 of the manuscript. It added Gaussian noise scaled to each feature's standard deviation to **every** feature, including the 0/1 flag and protocol indicators, without clipping, which produced physically impossible values (fractional or negative indicators and negative counts). It was therefore replaced by `3_Revised_Experiments.ipynb`, which perturbs only real-valued features with relative noise, clips at zero, uses 2-fold cross-validation inside the optimizer, class-balanced forests, and the archive consensus rule.
>
> Other differences from the final protocol: population 20 and 15 generations; 1,500 records per class with a single 70/30 validation split inside the optimizer; 15-tree inner forests; random forests without class weighting; the single three-objective Pareto solution as the proposed method (no consensus). It is kept for transparency only. Its output folder (`results_no_IAT/`) is not part of the reported results.

# Notebook 2 (WITHOUT IAT) — Multi-Objective Feature Selection for IoMT Cyberattack Detection (CICIoMT2024)

This notebook runs all the experiments for the manuscript, one simple step at a time.

**The idea in one sentence:** use fewer network features so the detector is lighter and faster, while still catching attacks well, including when the traffic is noisy.

| Step | What happens |
|---|---|
| 1 | Install and load tools, choose settings |
| 2 | Read the cleaned data from Notebook 1 |
| 3 | Look at the classes |
| 4 | Prepare the training and test sets |
| 5 | Define how a model is trained and scored |
| 6 | Define the feature-selection methods (MI, RFE, NSGA-II) |
| 7 | Run every method several times with different random seeds |
| 8 | Results table (mean ± 95% confidence interval) |
| 9 | Statistical tests |
| 10 | Stability of the selected features |
| 11 | Per-class results (rare attacks) and confusion matrix |
| 12 | Save everything |
| 13 | Figures |

**How to run:** first run **Notebook 1 (Data Cleaning)**, which creates the `cleaned_data` folder. Then open this notebook in the same folder and choose *Run All*. All outputs go into the `results_experiments/` folder.

## Step 1 — Tools and settings

`pymoo` provides the NSGA-II optimiser. The first cell installs it if it is missing.

You can change the settings in the second cell. The defaults finish in roughly 20–40 minutes on a normal laptop. To run a quick test first, set `N_RUNS = 3`.

**Recommended:** run the notebook twice: once with `EXCLUDE_IAT = False` and once with `EXCLUDE_IAT = True` (change `OUT` to e.g. `"results_no_IAT"` so the first results are not overwritten). IAT behaves like a capture timestamp, so the paper should show results both ways.

In [ ]:
import sys, subprocess
try:
    import pymoo
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymoo"])

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # prevents a common Windows crash (duplicate OpenMP library)
import re, glob, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             average_precision_score, confusion_matrix)
from sklearn.preprocessing import label_binarize
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize
warnings.filterwarnings("ignore")

# ------------------------- SETTINGS -------------------------
CLEAN_DIR = "cleaned_data"        # output folder of Notebook 1
OUT       = "results_no_IAT"
N_RUNS    = 10        # independent repetitions (different random seeds)
TRAIN_PER_CLASS = 20000   # training rows sampled per class in each run
TEST_PER_CLASS  = 20000   # test rows per class (same test set for every run)
NOISE_LEVEL = 0.10    # robustness test: Gaussian noise = 10% of each feature's std
GA_POP, GA_GEN = 20, 15   # NSGA-II population size and number of generations
GA_PER_CLASS = 1500       # rows per class used inside the optimiser (for speed)
F1_TOLERANCE = 0.01       # rule for picking one solution from the Pareto front
EXCLUDE_IAT  = True       # True = remove the timestamp-like IAT feature from the whole study
# -------------------------------------------------------------

os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "font.family": "serif", "axes.grid": True, "grid.alpha": 0.3})
print("Settings ready. Results folder:", os.path.abspath(OUT))

## Step 2 — Read the cleaned data

This notebook uses the output of **Notebook 1 (Data Cleaning)**: `cleaned_data/train_clean.csv` and `cleaned_data/test_clean.csv`. Missing and infinite values, impossible values, duplicate rows, label conflicts, train–test overlap, zero-variance features and duplicate features have already been handled there.

Train and test stay separate, as in the official dataset split.

In [ ]:
train_all = pd.read_csv(os.path.join(CLEAN_DIR, "train_clean.csv"))
test_all  = pd.read_csv(os.path.join(CLEAN_DIR, "test_clean.csv"))
FEATURES_ALL = [c for c in train_all.columns if c != "Label"]
train_all[FEATURES_ALL] = train_all[FEATURES_ALL].astype("float32")   # halves memory; done after cleaning
test_all[FEATURES_ALL]  = test_all[FEATURES_ALL].astype("float32")

CLASSES = sorted(set(train_all["Label"]) & set(test_all["Label"]))
print("Train rows:", f"{len(train_all):,}", "| Test rows:", f"{len(test_all):,}")
print("Features:", len(FEATURES_ALL), "| Classes:", CLASSES)

## Step 3 — The classes

Some attacks have far fewer rows than others. That is why we report **macro-F1**, which gives every class equal weight, and also look at each class separately (Step 11).

In [ ]:
def category(lbl):
    if lbl == "Benign": return "Benign"
    if "Spoofing" in lbl: return "Spoofing"
    if lbl.startswith("MQTT"):  return "MQTT"
    if lbl.startswith("Recon"): return "Recon"
    if "DDoS" in lbl: return "DDoS"
    if "DoS" in lbl:  return "DoS"
    return "Other"

class_table = pd.DataFrame({
    "Category": [category(c) for c in CLASSES],
    "Train rows (clean)": train_all["Label"].value_counts().reindex(CLASSES).values,
    "Test rows (clean)":  test_all["Label"].value_counts().reindex(CLASSES).values,
}, index=CLASSES)
class_table["Train rows used per run"] = np.minimum(class_table["Train rows (clean)"], TRAIN_PER_CLASS)
class_table["Test rows used"] = np.minimum(class_table["Test rows (clean)"], TEST_PER_CLASS)
class_table

## Step 4 — Prepare the data

1. Optionally remove IAT (see settings).
2. Build **one fixed test set** (same for every method and every run, so comparisons are fair).
3. Make a **noisy copy** of the test set to measure robustness.

In [ ]:
FEATURES = [c for c in FEATURES_ALL if not (EXCLUDE_IAT and c == "IAT")]
print("IAT excluded from the study:", EXCLUDE_IAT, "| Features used:", len(FEATURES))

def sample_per_class(df, label_col, n, seed):
    # take up to n random rows from each class (works on old and new pandas)
    idx = [g.sample(min(len(g), n), random_state=seed).index for _, g in df.groupby(label_col)]
    return df.loc[np.concatenate(idx)]

test_set = sample_per_class(test_all, "Label", TEST_PER_CLASS, seed=42)
X_test = test_set[FEATURES].values
y_test = test_set["Label"].values

# Noisy test copy: noise scale taken from the TRAINING data (no peeking at test)
feat_std = train_all[FEATURES].std().values
rng = np.random.default_rng(123)
X_test_noisy = X_test + rng.normal(0, 1, X_test.shape) * NOISE_LEVEL * feat_std

def sample_train(seed):
    s = sample_per_class(train_all, "Label", TRAIN_PER_CLASS, seed)
    return s[FEATURES].values, s["Label"].values

print("Test set size:", X_test.shape)

## Step 5 — How a model is trained and scored

Every feature subset is judged with the **same classifier**: a Random Forest with 100 trees.

| Metric | Meaning |
|---|---|
| Macro-F1 | balance of precision and recall, averaged equally over classes |
| Precision / Recall (macro) | how many alarms are correct / how many attacks are found |
| PR-AUC (macro) | area under the precision–recall curve |
| FPR | share of **benign** traffic wrongly flagged as attack (false alarms) |
| FNR | share of **attack** traffic wrongly called benign (missed attacks) |
| F1 under noise / F1 drop | performance on the noisy test set, and how much it fell |
| Train time, inference time | seconds to train; milliseconds to classify 1,000 records |
| Model size | memory of the saved model (MB) |

In [ ]:
def evaluate(feat_idx, X_tr, y_tr, seed):
    cols = list(feat_idx)
    model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)

    t0 = time.perf_counter(); model.fit(X_tr[:, cols], y_tr); train_time = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = model.predict(X_test[:, cols]); infer = time.perf_counter() - t0
    proba = model.predict_proba(X_test[:, cols])
    pred_noisy = model.predict(X_test_noisy[:, cols])

    benign = y_test == "Benign"
    f1 = f1_score(y_test, pred, average="macro")
    f1_noisy = f1_score(y_test, pred_noisy, average="macro")
    result = {
        "n_features": len(cols),
        "macro_F1": f1,
        "precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "PR_AUC": average_precision_score(label_binarize(y_test, classes=model.classes_), proba, average="macro"),
        "FPR": np.mean(pred[benign] != "Benign"),
        "FNR": np.mean(pred[~benign] == "Benign"),
        "F1_noisy": f1_noisy,
        "F1_drop_noise": f1 - f1_noisy,
        "train_time_s": train_time,
        "infer_ms_per_1000": 1000 * infer / len(X_test) * 1000,
        "model_MB": len(pickle.dumps(model)) / 1e6,
    }
    per_class = f1_score(y_test, pred, average=None, labels=CLASSES)
    return result, dict(zip(CLASSES, per_class)), pred

## Step 6 — The feature-selection methods

| Method | How it chooses features |
|---|---|
| **All features** | no selection (reference) |
| **All without IAT** | ablation: is the timestamp-like IAT feature driving results? |
| **Mutual Information (MI)** | ranks features by how much they tell us about the label; keeps the top *k* |
| **RFE** | repeatedly removes the least important feature of a Random Forest until *k* remain |
| **NSGA-II (2 objectives)** | genetic search that minimises (1) error and (2) number of features |
| **NSGA-II (3 objectives) – proposed** | as above, plus (3) the drop in F1 under noise (robustness) |

**Fair comparison:** MI and RFE are given the **same number of features *k*** that the proposed method chose in that run.

**How NSGA-II works (simply):** each candidate is a list of 0/1 switches (1 = use the feature). A population of candidates is scored, the best are combined and slightly mutated, and this repeats. The result is a **Pareto front**: a set of solutions where no objective can be improved without worsening another.

**Picking one solution:** among the Pareto solutions whose validation F1 is within 0.01 of the best, we take the one with the **fewest features** (ties: smallest noise drop).

To avoid leakage, NSGA-II scores candidates on a **validation split of the training data only**. The test set is never used during selection.

In [ ]:
class FeatureSelectionProblem(ElementwiseProblem):
    def __init__(self, Xa, ya, Xv, yv, Xv_noisy, n_obj, seed):
        super().__init__(n_var=Xa.shape[1], n_obj=n_obj, xl=0, xu=1, vtype=bool)
        self.Xa, self.ya, self.Xv, self.yv, self.Xvn, self.seed = Xa, ya, Xv, yv, Xv_noisy, seed

    def _evaluate(self, x, out, *args, **kwargs):
        idx = np.where(x)[0]
        if len(idx) == 0:                       # empty subset = worst possible
            out["F"] = [1.0, 0.0, 1.0][: self.n_obj]; return
        clf = RandomForestClassifier(n_estimators=15, random_state=self.seed, n_jobs=-1)
        clf.fit(self.Xa[:, idx], self.ya)
        f1 = f1_score(self.yv, clf.predict(self.Xv[:, idx]), average="macro")
        F = [1 - f1, len(idx) / self.n_var]
        if self.n_obj == 3:
            f1n = f1_score(self.yv, clf.predict(self.Xvn[:, idx]), average="macro")
            F.append(max(0.0, f1 - f1n))
        out["F"] = F

def nsga2_select(X_tr, y_tr, seed, n_obj):
    # small, balanced subset of the training data, split into fit / validation
    df = pd.DataFrame(X_tr); df["y"] = y_tr
    sub = sample_per_class(df, "y", GA_PER_CLASS, seed)
    Xs, ys = sub.drop(columns="y").values, sub["y"].values
    Xa, Xv, ya, yv = train_test_split(Xs, ys, test_size=0.3, stratify=ys, random_state=seed)
    Xv_noisy = Xv + np.random.default_rng(seed).normal(0, 1, Xv.shape) * NOISE_LEVEL * feat_std

    problem = FeatureSelectionProblem(Xa, ya, Xv, yv, Xv_noisy, n_obj, seed)
    algo = NSGA2(pop_size=GA_POP, sampling=BinaryRandomSampling(),
                 crossover=TwoPointCrossover(), mutation=BitflipMutation(),
                 eliminate_duplicates=True)
    res = minimize(problem, algo, ("n_gen", GA_GEN), seed=seed, verbose=False)

    F, Xsol = np.atleast_2d(res.F), np.atleast_2d(res.X)
    val_f1 = 1 - F[:, 0]
    ok = np.where(val_f1 >= val_f1.max() - F1_TOLERANCE)[0]
    tie = F[ok, 2] if n_obj == 3 else np.zeros(len(ok))
    best = ok[np.lexsort((tie, F[ok, 1]))[0]]
    return np.where(Xsol[best])[0], F

def mi_select(X_tr, y_tr, k, seed):
    idx = np.random.default_rng(seed).choice(len(X_tr), min(20000, len(X_tr)), replace=False)
    scores = mutual_info_classif(X_tr[idx], y_tr[idx], random_state=seed)
    return np.argsort(scores)[::-1][:k]

def rfe_select(X_tr, y_tr, k, seed):
    idx = np.random.default_rng(seed).choice(len(X_tr), min(10000, len(X_tr)), replace=False)
    rfe = RFE(RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1),
              n_features_to_select=k, step=2).fit(X_tr[idx], y_tr[idx])
    return np.where(rfe.support_)[0]

ALL_IDX = np.arange(len(FEATURES))
NO_IAT_IDX = np.array([i for i, f in enumerate(FEATURES) if f != "IAT"])
print("Methods defined.")

## Step 7 — Run the experiments

For each run (seed): draw a new training sample → run every method → train the final model → score it on the fixed test set. Progress is printed as it goes.

**Progress is saved after every run** (file `checkpoint.pkl` in the results folder). If the notebook is interrupted (restart, sleep, closed tab), just choose **Run All** again: finished runs are loaded from the file and the experiment continues where it stopped. If you change any setting, it automatically starts fresh.

In [ ]:
CHECKPOINT = os.path.join(OUT, "checkpoint.pkl")
signature = (TRAIN_PER_CLASS, TEST_PER_CLASS, NOISE_LEVEL, GA_POP, GA_GEN, GA_PER_CLASS,
             F1_TOLERANCE, EXCLUDE_IAT, tuple(FEATURES), tuple(CLASSES))

rows, per_class_rows, selected = [], [], []
pareto_example, preds_example = None, None
first_run = 0
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, "rb") as fh:
        ck = pickle.load(fh)
    if ck["signature"] == signature:
        rows, per_class_rows, selected = ck["rows"], ck["per_class_rows"], ck["selected"]
        pareto_example, preds_example, first_run = ck["pareto_example"], ck["preds_example"], ck["next_run"]
        print(f"Saved progress found: {first_run} run(s) already finished - continuing from run {first_run + 1}.")
    else:
        print("Settings changed since the saved progress - starting fresh.")

t_start = time.time()
for run in range(first_run, N_RUNS):
    seed = run
    X_tr, y_tr = sample_train(seed)

    t0 = time.perf_counter(); sel3, F3 = nsga2_select(X_tr, y_tr, seed, n_obj=3); t_n3 = time.perf_counter() - t0
    t0 = time.perf_counter(); sel2, _  = nsga2_select(X_tr, y_tr, seed, n_obj=2); t_n2 = time.perf_counter() - t0
    k = len(sel3)
    t0 = time.perf_counter(); sel_mi  = mi_select(X_tr, y_tr, k, seed);  t_mi  = time.perf_counter() - t0
    t0 = time.perf_counter(); sel_rfe = rfe_select(X_tr, y_tr, k, seed); t_rfe = time.perf_counter() - t0

    methods = {
        "All features":        (ALL_IDX, 0.0),
        "MI (top-k)":          (sel_mi, t_mi),
        "RFE (top-k)":         (sel_rfe, t_rfe),
        "NSGA-II (2-obj)":     (sel2, t_n2),
        "NSGA-II (3-obj, proposed)": (sel3, t_n3),
    }
    if "IAT" in FEATURES:                       # ablation only makes sense if IAT is present
        methods["All without IAT"] = (NO_IAT_IDX, 0.0)
    for name, (idx, sel_time) in methods.items():
        res, pc, pred = evaluate(idx, X_tr, y_tr, seed)
        rows.append({"run": run, "method": name, "selection_time_s": sel_time, **res})
        per_class_rows.append({"run": run, "method": name, **pc})
        selected.append({"run": run, "method": name, "features": [FEATURES[i] for i in idx]})
        if run == 0 and name == "NSGA-II (3-obj, proposed)":
            pareto_example, preds_example = F3, pred

    with open(CHECKPOINT, "wb") as fh:            # save progress after every run
        pickle.dump({"signature": signature, "rows": rows, "per_class_rows": per_class_rows,
                     "selected": selected, "pareto_example": pareto_example,
                     "preds_example": preds_example, "next_run": run + 1}, fh)
    print(f"Run {run+1}/{N_RUNS} done  | k = {k} features | elapsed {(time.time()-t_start)/60:.1f} min | progress saved")

results = pd.DataFrame(rows)
per_class = pd.DataFrame(per_class_rows)
selected = pd.DataFrame(selected)
print("\nAll runs finished.")

## Step 8 — Main results table

Each value is the **mean ± half-width of the 95% confidence interval** over all runs (t-distribution).

In [ ]:
METHOD_ORDER = [m for m in ["All features", "All without IAT", "MI (top-k)", "RFE (top-k)",
                "NSGA-II (2-obj)", "NSGA-II (3-obj, proposed)"] if m in set(results["method"])]
METRICS = ["n_features", "macro_F1", "precision", "recall", "PR_AUC", "FPR", "FNR",
           "F1_noisy", "F1_drop_noise", "selection_time_s", "train_time_s",
           "infer_ms_per_1000", "model_MB"]

def ci95(x):
    x = np.asarray(x, float)
    if len(x) < 2: return 0.0
    return stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))

mean_t = results.groupby("method")[METRICS].mean().reindex(METHOD_ORDER)
ci_t   = results.groupby("method")[METRICS].agg(ci95).reindex(METHOD_ORDER)

summary = mean_t.copy().astype(object)
for m in METRICS:
    dec = 2 if m in ("n_features", "selection_time_s", "train_time_s", "infer_ms_per_1000", "model_MB") else 4
    summary[m] = [f"{a:.{dec}f} ± {b:.{dec}f}" for a, b in zip(mean_t[m], ci_t[m])]
summary

## Step 9 — Statistical tests

**Question:** is the proposed method's result really different from each other method, or could it be chance?

We use the **Wilcoxon signed-rank test**. It pairs results from the same run and does not assume normally distributed data. A p-value below 0.05 indicates a significant difference. We use **Holm correction** because several comparisons are made at once.

A **Friedman test** first checks whether *any* difference exists among all methods.

*Note:* with 10 runs the Wilcoxon test can detect differences; with fewer than 6 runs it cannot reach p < 0.05.

In [ ]:
def holm(pvals):
    p = np.asarray(pvals); order = np.argsort(p); adj = np.empty_like(p); m = len(p); running = 0
    for rank, i in enumerate(order):
        running = max(running, min(1, (m - rank) * p[i])); adj[i] = running
    return adj

stat_rows = []
for metric in ["macro_F1", "F1_noisy", "F1_drop_noise", "n_features", "infer_ms_per_1000"]:
    wide = results.pivot(index="run", columns="method", values=metric)[METHOD_ORDER]
    fr = stats.friedmanchisquare(*[wide[c] for c in METHOD_ORDER])
    prop = wide["NSGA-II (3-obj, proposed)"]
    comps = []
    for other in METHOD_ORDER[:-1]:
        d = prop - wide[other]
        p = 1.0 if np.allclose(d, 0) else stats.wilcoxon(prop, wide[other]).pvalue
        comps.append((other, d.mean(), p))
    adj = holm([c[2] for c in comps])
    for (other, diff, p), pa in zip(comps, adj):
        stat_rows.append({"metric": metric, "Friedman p": fr.pvalue, "proposed vs": other,
                          "mean difference (proposed - other)": diff,
                          "Wilcoxon p": p, "Holm-adjusted p": pa,
                          "significant (0.05)": "yes" if pa < 0.05 else "no"})
stats_table = pd.DataFrame(stat_rows)
stats_table.round(5)

## Step 10 — Are the same features chosen every time? (Stability)

A good method should pick **similar features** when the training sample changes slightly. We compare every pair of runs with the **Jaccard index** = (features in both) / (features in either). A value of 1 means identical sets; 0 means nothing in common.

In [ ]:
def mean_jaccard(sets):
    vals = [len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:] if a | b]
    return np.mean(vals) if vals else np.nan

stab = []
for m in ["MI (top-k)", "RFE (top-k)", "NSGA-II (2-obj)", "NSGA-II (3-obj, proposed)"]:
    sets = [set(f) for f in selected[selected["method"] == m]["features"]]
    stab.append({"method": m, "mean Jaccard stability": mean_jaccard(sets),
                 "mean subset size": np.mean([len(s) for s in sets])})
stability = pd.DataFrame(stab)

# How often each feature was chosen by the proposed method
prop_sets = selected[selected["method"] == "NSGA-II (3-obj, proposed)"]["features"]
freq = pd.Series([f for s in prop_sets for f in s]).value_counts().reindex(FEATURES, fill_value=0)
freq = (freq / N_RUNS).sort_values(ascending=False).rename("selection frequency")
display(stability.round(4))
freq.head(20)

## Step 11 — Results per class (are rare attacks still detected?)

The table shows the **F1-score for each class** (mean over runs). Rare classes such as ARP spoofing and Recon-VulScan are the ones most at risk when features are removed.

In [ ]:
per_class_mean = per_class.groupby("method")[CLASSES].mean().reindex(METHOD_ORDER).T
per_class_mean.insert(0, "Test rows", class_table["Test rows used"])
per_class_mean.round(4)

## Step 12 — Save everything (before drawing figures)

- `results_experiments/experiment_results.xlsx` – every table, one per sheet
- `results_experiments/summary_for_manuscript.txt` – the key results in plain text (**send this file back** so we can write the Results section)

Results are saved **before** the figures are drawn, so nothing is lost even if drawing a figure fails.

In [ ]:
with pd.ExcelWriter(f"{OUT}/experiment_results.xlsx") as xw:
    class_table.to_excel(xw, sheet_name="Dataset")
    summary.to_excel(xw, sheet_name="Summary_mean_CI")
    mean_t.to_excel(xw, sheet_name="Means")
    stats_table.to_excel(xw, sheet_name="Statistical_tests", index=False)
    stability.to_excel(xw, sheet_name="Stability", index=False)
    freq.to_frame().to_excel(xw, sheet_name="Feature_frequency")
    per_class_mean.to_excel(xw, sheet_name="Per_class_F1")
    results.to_excel(xw, sheet_name="All_runs", index=False)
    selected.assign(features=selected["features"].apply(", ".join)).to_excel(xw, sheet_name="Selected_features", index=False)
    pd.DataFrame({"n_features": pareto_example[:, 1] * len(FEATURES), "val_macro_F1": 1 - pareto_example[:, 0],
                  "F1_drop_noise": pareto_example[:, 2]}).to_excel(xw, sheet_name="Pareto_front_run1", index=False)
    pd.DataFrame(confusion_matrix(y_test, preds_example, labels=CLASSES, normalize="true"),
                 index=CLASSES, columns=CLASSES).to_excel(xw, sheet_name="Confusion_matrix_run1")

pd.set_option("display.width", 250)
lines = ["EXPERIMENT SUMMARY — CICIoMT2024 multi-objective feature selection", "=" * 70,
         f"Runs: {N_RUNS} | train rows/class: {TRAIN_PER_CLASS} | test rows/class: {TEST_PER_CLASS}",
         f"NSGA-II: pop={GA_POP}, gen={GA_GEN}, rows/class inside optimiser={GA_PER_CLASS}, noise={NOISE_LEVEL}",
         f"Classes used: {', '.join(CLASSES)}",
         f"IAT excluded: {EXCLUDE_IAT}",
         f"Features used: {len(FEATURES)}",
         "\nDataset:\n" + class_table.to_string(),
         "\nMAIN RESULTS (mean ± 95% CI):\n" + summary.T.to_string(),
         "\nSTATISTICAL TESTS:\n" + stats_table.round(5).to_string(index=False),
         "\nSTABILITY:\n" + stability.round(4).to_string(index=False),
         "\nFEATURE SELECTION FREQUENCY (proposed):\n" + freq[freq > 0].round(2).to_string(),
         "\nPER-CLASS F1 (mean):\n" + per_class_mean.round(4).to_string()]
text = "\n".join(lines)
with open(f"{OUT}/summary_for_manuscript.txt", "w", encoding="utf-8") as fh:
    fh.write(text)
print(text)
print("\nSaved:", sorted(os.listdir(OUT)))

## Step 13 — Figures

All results are already saved (Step 12). Each figure is drawn in its own cell. If one cell fails, the others and the saved results are not affected.

In [ ]:
short = {"All features": "All", "All without IAT": "All−IAT", "MI (top-k)": "MI",
         "RFE (top-k)": "RFE", "NSGA-II (2-obj)": "NSGA-II\n2-obj", "NSGA-II (3-obj, proposed)": "NSGA-II\n3-obj"}
COLOR = {"All features": "#7F7F7F", "All without IAT": "#BFBFBF", "MI (top-k)": "#5B9BD5",
         "RFE (top-k)": "#ED7D31", "NSGA-II (2-obj)": "#A5A5A5", "NSGA-II (3-obj, proposed)": "#1F4E78"}
colors = [COLOR[m] for m in METHOD_ORDER]
labels = [short[m] for m in METHOD_ORDER]

In [ ]:
# Figure A: macro-F1 clean vs noisy
fig, ax = plt.subplots(figsize=(7, 3.8)); x = np.arange(len(METHOD_ORDER)); w = 0.38
ax.bar(x - w/2, mean_t["macro_F1"], w, yerr=ci_t["macro_F1"], label="Clean test", color="#1F4E78", capsize=3)
ax.bar(x + w/2, mean_t["F1_noisy"], w, yerr=ci_t["F1_noisy"], label=f"Noisy test ({int(NOISE_LEVEL*100)}% noise)", color="#ED7D31", capsize=3)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8); ax.set_ylabel("Macro-F1")
ax.set_ylim(max(0, min(mean_t["F1_noisy"].min(), mean_t["macro_F1"].min()) - 0.05), 1.0)
ax.legend(fontsize=8); ax.set_title("Figure A. Detection performance, clean vs noisy (mean ± 95% CI)")
plt.tight_layout(); plt.savefig(f"{OUT}/figA_f1_clean_vs_noisy.png"); plt.show()
plt.close("all")

In [ ]:
# Figure B: trade-off between number of features and macro-F1
fig, ax = plt.subplots(figsize=(6, 4))
for m, c in zip(METHOD_ORDER, colors):
    ax.errorbar(mean_t.loc[m, "n_features"], mean_t.loc[m, "macro_F1"],
                xerr=ci_t.loc[m, "n_features"], yerr=ci_t.loc[m, "macro_F1"],
                fmt="o", color=c, capsize=3, markersize=8, label=short[m].replace("\n", " "))
ax.set_xlabel("Number of features"); ax.set_ylabel("Macro-F1"); ax.legend(fontsize=8)
ax.set_title("Figure B. Accuracy vs. number of features")
plt.tight_layout(); plt.savefig(f"{OUT}/figB_tradeoff.png"); plt.show()
plt.close("all")

In [ ]:
# Figure C: inference time and model size
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, met, lab in zip(axes, ["infer_ms_per_1000", "model_MB"], ["Inference time (ms / 1,000 records)", "Model size (MB)"]):
    ax.bar(labels, mean_t[met], yerr=ci_t[met], color=colors, capsize=3)
    ax.set_ylabel(lab); ax.tick_params(axis="x", labelsize=7)
fig.suptitle("Figure C. Computational cost"); plt.tight_layout()
plt.savefig(f"{OUT}/figC_cost.png"); plt.show()
plt.close("all")

In [ ]:
# Figure D: Pareto front from run 1
F = pareto_example
fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(F[:, 1] * len(FEATURES), 1 - F[:, 0], c=F[:, 2], cmap="viridis_r", s=50, edgecolor="k")
plt.colorbar(sc, label="F1 drop under noise (lower = more robust)")
ax.set_xlabel("Number of features"); ax.set_ylabel("Validation macro-F1")
ax.set_title("Figure D. NSGA-II Pareto front (run 1)")
plt.tight_layout(); plt.savefig(f"{OUT}/figD_pareto_front.png"); plt.show()
plt.close("all")

In [ ]:
# Figure E: how often each feature is selected by the proposed method
top = freq[freq > 0]
fig, ax = plt.subplots(figsize=(6, max(3, 0.25 * len(top))))
ax.barh(top.index[::-1], top.values[::-1], color="#1F4E78")
ax.set_xlabel("Share of runs in which the feature was selected"); ax.set_xlim(0, 1)
ax.set_title("Figure E. Features chosen by the proposed method")
plt.tight_layout(); plt.savefig(f"{OUT}/figE_feature_frequency.png"); plt.show()
plt.close("all")

In [ ]:
# Figure F: confusion matrix of the proposed method (run 1), row-normalised
cm = confusion_matrix(y_test, preds_example, labels=CLASSES, normalize="true")
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); plt.colorbar(im, ax=ax)
ax.set_title("Figure F. Confusion matrix, proposed method (run 1)")
plt.tight_layout(); plt.savefig(f"{OUT}/figF_confusion_matrix.png"); plt.show()
plt.close("all")